In [2]:
import pandas as pd
from collections import Counter
import os
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import pickle

In [3]:
bucket = os.getenv("WORKSPACE_BUCKET")

In [3]:
# !pip install ftfy

In [4]:
def write_df_to_bucket(df, filename: str):
    # save mt directly to bucket
    df.to_csv(f"{bucket}/data/{filename}.csv", index = None)
def read_df_from_bucket(filename: str):
    # save mt directly to bucket
    return pd.read_csv(f"{bucket}/data/{filename}.csv")

In [5]:
import hail as hl
bucket = os.getenv("WORKSPACE_BUCKET")

Loading BokehJS ...

In [6]:
hl.init(default_reference='GRCh38', idempotent=True)
mt_clinvar_path = os.getenv("WGS_CLINVAR_SPLIT_HAIL_PATH")
mt = hl.read_matrix_table(mt_clinvar_path) # mt.count() #old (1281259, 245394) new (2180727, 414830)

/opt/conda/lib/python3.10/site-packages/hail/context.py:350: UserWarning:

Using hl.init with a default_reference argument is deprecated. To set a default reference genome after initializing hail, call `hl.default_reference` with an argument to set the default reference genome.

/opt/conda/lib/python3.10/site-packages/hailtop/aiocloud/aiogoogle/user_config.py:43: UserWarning:

Reading spark-defaults.conf to determine GCS requester pays configuration. This is deprecated. Please use `hailctl config set gcs_requester_pays/project` and `hailctl config set gcs_requester_pays/buckets`.

Running on Apache Spark version 3.5.3
SparkUI available at http://all-of-us-9013-m.us-central1-f.c.terra-vpc-sc-e9211784.internal:37161
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.134-952ae203dbbe
LOGGING: writing to /home/jupyter/workspaces/ehrcancermodelevaluation/hail-20250711-1608-0.2.134-952ae203dbbe.log


In [7]:
mt.count()

(2180727, 414830)

In [7]:
related_samples_path = "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/relatedness/relatedness_flagged_samples.tsv"
related_remove = hl.import_table(related_samples_path,
                                 types={"sample_id":"tstr"},
                                key="sample_id")

In [8]:
mt = mt.anti_join_cols(related_remove) #(2180727, 384246)
mt.count()

(2180727, 384246)

In [11]:
with open('pc_genetic_variant.pickle', 'rb') as f:
    clinvar_variant = pickle.load(f)

In [4]:
gene = ['ATM', 'BRCA1', 'BRCA2', 'FAMMM', 'PALB2', 'EPCAM', 'MLH1', 'MSH2', 'MSH6', 'PMS2', 'PRSS1', 'PRSS2', 'PRSS1/2','CTRC', 'STK11']
gene_ = []
for i in gene:
    print(Counter(clinvar_variant[i]['#CHROM']))
    if len(Counter(clinvar_variant[i]['#CHROM']))>0:
        gene_.append(i)

In [15]:
def identify_pat(g):
    mt_collect = []
    chromosome = g['#CHROM'].values[0]
    loci_of_interest = {hl.parse_locus("chr" + str(chromosome) + ":" + str(k), reference_genome='GRCh38') for k in g['POS']}
    mt_chromosome = mt.filter_rows(hl.literal(loci_of_interest).contains(mt.locus))

    g_pat = []

#     for i, row in tqdm(g.iterrows(), total = g.shape[0]):
    for i, row in g.iterrows():
        position = row['POS']  # Corrected one-based position

        ref_allele = row['REF']
        alt_allele = row['ALT']


        filtered_mt = mt_chromosome.filter_rows((mt_chromosome.locus.position == position) & 
                                                (mt_chromosome.alleles[0] == ref_allele) & 
                                                (mt_chromosome.alleles[1] == alt_allele))

        # Identify patients (samples) that have a non-reference genotype for this variant
        patients_with_mutation = filtered_mt.filter_cols(hl.agg.any(filtered_mt.GT.is_non_ref()))

        # Collect the sample IDs of the patients
        patient_ids = patients_with_mutation.s.collect()

        if len(patient_ids)>0:
            g_pat = g_pat + patient_ids
    return g_pat

In [16]:
chunk_size = 100

In [17]:
gene = ['BRCA2', 'FAMMM', 'PALB2', 'EPCAM', 'MLH1', 'MSH2', 'MSH6', 'PMS2', 'PRSS1', 'PRSS2', 'PRSS1/2','CTRC', 'STK11']

In [5]:
carriers = {}
for i in gene:
    g = clinvar_variant[i]
    if len(g)>0:
        carriers[i] = []
        chunks = np.array_split(g, int(np.ceil(len(g) / chunk_size)))
        for chunk in tqdm(chunks):
            carriers[i].extend(identify_pat(chunk))
            with open('carriers_'+i+'.pickle', 'wb') as f:
                pickle.dump(carriers[i], f)
    print(i, ' done')

In [23]:
carriers_dict = {}
for i in gene_:
    with open('carriers_'+i+'.pickle', 'rb') as f:
        carriers_dict[i] = pickle.load(f)

In [26]:
carriers_df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in carriers_dict.items()]))

In [28]:
write_df_to_bucket(carriers_df, 'pc_gene_carriers')